# ANKA–DMM Haber Sınıflandırma Modeli

Bu notebook, ANKA haber metinleri ile DMM tarafından yanlış olarak etiketlenmiş iddiaları ayırmak için bir **TF-IDF + Logistic Regression** modeli eğitir.

> **Not:** Model, bir haberin doğruluğunu dış kaynaklardan doğrulamaz. ANKA ve DMM veri kümelerindeki metinsel örüntüleri öğrenir.


In [ ]:
# Notebook için gerekli kütüphaneleri kurar.
!pip install -q -U polars pyarrow scikit-learn pandas matplotlib


## 1. Dosya yolları ve Google Drive

Google Drive bağlanır; Drive üzerindeki ANKA ve DMM Parquet dosyalarının kaynak yolları ile Colab çalışma alanındaki hedef yollar tanımlanır.


In [ ]:
# Google Drive bağlantısını kurar ve veri dosyalarının yollarını tanımlar.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ANKA_DRIVE_PATH = Path(
    "/content/drive/MyDrive/datasets/anka_ajansi_haberler.parquet"
)

DMM_DRIVE_PATH = Path(
    "/content/drive/MyDrive/datasets/ddm.parquet"
)

ANKA_PATH = Path(
    "/content/anka_ajansi_haberler.parquet"
)

DMM_PATH = Path(
    "/content/dmm.parquet"
)


## 2. Verileri Colab çalışma alanına kopyalama

Dosyalar daha hızlı yerel erişim için Drive'dan `/content` dizinine kopyalanır.


In [ ]:
# Drive üzerindeki dosyaları Colab'ın yerel çalışma alanına kopyalar.
import shutil

for source, target in [
    (ANKA_DRIVE_PATH, ANKA_PATH),
    (DMM_DRIVE_PATH, DMM_PATH),
]:
    if not source.exists():
        raise FileNotFoundError(
            f"Drive dosyası bulunamadı: {source}"
        )

    shutil.copy2(source, target)

    print(
        f"Kopyalandı: {source.name} -> {target}"
    )
    print(
        f"Boyut: {target.stat().st_size / 1024**2:.2f} MB"
    )


## 3. Parquet dosyalarını doğrulama

Her dosyanın varlığı, minimum boyutu ve Parquet imzası (`PAR1`) kontrol edilir.


In [ ]:
# Dosyaların geçerli ve eksiksiz Parquet dosyaları olduğunu kontrol eder.
def validate_parquet(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)

    if path.stat().st_size < 8:
        raise ValueError(
            f"Dosya çok küçük: {path}"
        )

    with path.open("rb") as file:
        start = file.read(4)
        file.seek(-4, 2)
        end = file.read(4)

    print(
        path.name,
        "| başlangıç:",
        start,
        "| bitiş:",
        end,
    )

    if start != b"PAR1" or end != b"PAR1":
        raise ValueError(
            f"Geçersiz veya eksik Parquet: {path}"
        )


validate_parquet(ANKA_PATH)
validate_parquet(DMM_PATH)


## 4. Veri kümelerini yükleme ve inceleme

Parquet dosyaları Polars ile okunur; boyutları, sütunları ve DMM etiket dağılımı incelenir.


In [ ]:
# ANKA ve DMM veri kümelerini Polars DataFrame olarak yükler.
import polars as pl

anka_raw = pl.read_parquet(ANKA_PATH)
dmm_raw = pl.read_parquet(DMM_PATH)

print("ANKA shape:", anka_raw.shape)
print("ANKA columns:", anka_raw.columns)

print()

print("DMM shape:", dmm_raw.shape)
print("DMM columns:", dmm_raw.columns)


In [ ]:
# DMM veri kümesindeki etiketlerin örnek sayılarını gösterir.
dmm_raw.group_by(
    "rating_label"
).len().sort(
    "len",
    descending=True,
)


## 5. DMM yanlış iddia sınıfını hazırlama

DMM verisinden `Yanlış` etiketli kayıtlar seçilir. Model girdisi olarak yalnızca `claim` alanı kullanılır; tekrarlar ve çok kısa metinler kaldırılır.

- `label = 1`
- `source = dmm_false_claim`


In [ ]:
# DMM'deki yanlış iddialardan pozitif sınıfı (label=1) oluşturur.
fake_df = (
    dmm_raw
    .filter(
        pl.col("rating_label")
        .cast(pl.String)
        .str.to_lowercase()
        .str.contains("yanlış")
    )
    .select([
        pl.col("id")
        .cast(pl.String)
        .alias("id"),

        pl.col("claim")
        .cast(pl.String)
        .str.strip_chars()
        .alias("text"),
    ])
    .drop_nulls(["text"])
    .filter(
        pl.col("text").str.len_chars() >= 20
    )
    .unique(
        subset=["text"],
        maintain_order=True,
    )
    .with_columns([
        pl.lit(1).alias("label"),
        pl.lit("dmm_false_claim").alias("source"),
    ])
)

print("DMM sahte/yanlış:", fake_df.shape)

fake_df.head(5)


In [ ]:
# DMM veri kümesinde bulunan benzersiz etiket adlarını listeler.
print(
    dmm_raw["rating_label"]
    .drop_nulls()
    .unique()
    .to_list()
)


In [ ]:
# Etiketi tam olarak 'Yanlış' olan kayıtlarla fake_df değişkenini yeniden oluşturur.
fake_df = (
    dmm_raw
    .filter(
        pl.col("rating_label") == "Yanlış"
    )
    .select([
        pl.col("id").cast(pl.String).alias("id"),
        pl.col("claim").cast(pl.String).alias("text"),
    ])
    .drop_nulls(["text"])
    .filter(
        pl.col("text").str.len_chars() >= 20
    )
    .unique(subset=["text"])
    .with_columns([
        pl.lit(1).alias("label"),
        pl.lit("dmm_false_claim").alias("source"),
    ])
)


## 6. ANKA gerçek haber sınıfını hazırlama

ANKA veri setindeki uygun sütunlar otomatik seçilir. Başlık ve özet birleştirilerek model girdisi oluşturulur.

- `label = 0`
- `source = anka_real`


In [ ]:
# ANKA veri kümesindeki mevcut sütun adlarını gösterir.
print(anka_raw.columns)


In [ ]:
# Farklı veri sürümlerinde kullanılabilecek uygun ANKA sütunlarını seçer.
def choose_column(
    dataframe: pl.DataFrame,
    candidates: list[str],
) -> str:
    for column in candidates:
        if column in dataframe.columns:
            return column

    raise KeyError(
        f"Şu sütunlardan hiçbiri bulunamadı: {candidates}"
    )


title_col = choose_column(
    anka_raw,
    ["Baslik"],
)

summary_col = choose_column(
    anka_raw,
    ["Ozet_Temiz", "Ozet"],
)

body_col = choose_column(
    anka_raw,
    ["Detay_Metin_Temiz", "Detay_Metin"],
)

id_col = choose_column(
    anka_raw,
    ["UUID", "id"],
)

print("Başlık:", title_col)
print("Özet:", summary_col)
print("Metin:", body_col)
print("ID:", id_col)


In [ ]:
# ANKA başlık ve özetlerinden negatif sınıfı (label=0) oluşturur.
real_df = (
    anka_raw
    .select([
        pl.col(id_col)
        .cast(pl.String)
        .alias("id"),

        pl.concat_str(
            [
                pl.col(title_col)
                .cast(pl.String)
                .fill_null("")
                .str.strip_chars(),

                pl.col(summary_col)
                .cast(pl.String)
                .fill_null("")
                .str.strip_chars(),
            ],
            separator=" ",
        )
        .str.strip_chars()
        .alias("text"),
    ])
    .drop_nulls(["text"])
    .filter(
        pl.col("text").str.len_chars() >= 20
    )
    .unique(
        subset=["text"],
        maintain_order=True,
    )
    .with_columns([
        pl.lit(0).alias("label"),
        pl.lit("anka_real").alias("source"),
    ])
)

print("ANKA gerçek:", real_df.shape)

real_df.head(5)


## 7. Metin uzunluklarını inceleme ve sınırlandırma

İki kaynağın kelime uzunluğu dağılımları karşılaştırılır. Çok uzun metinlerin yalnızca ilk `MAX_WORDS` kelimesi korunur.


In [ ]:
# Her sınıftaki metinlerin kelime sayısı dağılımını özetler.
def word_length_stats(
    dataframe: pl.DataFrame,
    name: str,
):
    lengths = dataframe.select(
        pl.col("text")
        .str.split(" ")
        .list.eval(
            pl.element().filter(
                pl.element().str.len_chars() > 0
            )
        )
        .list.len()
        .alias("word_count")
    )

    print(name)
    print(lengths["word_count"].describe())


word_length_stats(
    real_df,
    "ANKA gerçek",
)

word_length_stats(
    fake_df,
    "DMM yanlış",
)


In [ ]:
# Her metni en fazla MAX_WORDS kelimeyle sınırlar.
MAX_WORDS = 120


def truncate_words(text: str, max_words: int) -> str:
    if text is None:
        return ""

    return " ".join(
        str(text).split()[:max_words]
    )


real_df = real_df.with_columns(
    pl.col("text")
    .map_elements(
        lambda value: truncate_words(
            value,
            MAX_WORDS,
        ),
        return_dtype=pl.String,
    )
    .alias("text")
)

fake_df = fake_df.with_columns(
    pl.col("text")
    .map_elements(
        lambda value: truncate_words(
            value,
            MAX_WORDS,
        ),
        return_dtype=pl.String,
    )
    .alias("text")
)


## 8. Sınıfları dengeleme ve birleşik veri setini kaydetme

Azınlık sınıfının örnek sayısı temel alınarak iki sınıf eşitlenir. Birleşik veri seti hem `/content` dizinine hem de Drive'a kaydedilir.


In [ ]:
# İki sınıfı eşit örnek sayısına getirir ve tek veri setinde birleştirir.
n_samples = min(
    len(real_df),
    len(fake_df),
)

print("Her sınıftan kullanılacak:", n_samples)

real_balanced = real_df.sample(
    n=n_samples,
    shuffle=True,
    seed=42,
)

fake_balanced = fake_df.sample(
    n=n_samples,
    shuffle=True,
    seed=42,
)

dataset_df = pl.concat([
    real_balanced,
    fake_balanced,
]).sample(
    fraction=1.0,
    shuffle=True,
    seed=42,
)

print(dataset_df.shape)

print(
    dataset_df
    .group_by(["label", "source"])
    .len()
)


In [ ]:
# Birleşik veri setini Colab çalışma alanına kaydeder.
COMBINED_PATH = Path(
    "/content/anka_dmm_real_fake_dataset.parquet"
)

dataset_df.write_parquet(
    COMBINED_PATH
)

print("Kaydedildi:", COMBINED_PATH)


In [ ]:
# Birleşik veri setinin kalıcı bir kopyasını Google Drive'a kaydeder.
DRIVE_COMBINED_PATH = Path(
    "/content/drive/MyDrive/anka_dmm_real_fake_dataset.parquet"
)

dataset_df.write_parquet(
    DRIVE_COMBINED_PATH
)

print(
    "Drive'a kaydedildi:",
    DRIVE_COMBINED_PATH,
)


## 9. Train–validation–test ayrımı

Birleşik veri seti sınıf oranları korunarak:

- `%80` train
- `%10` validation
- `%10` test

olarak ayrılır.


In [ ]:
# Veriyi stratified biçimde train, validation ve test kümelerine ayırır.
from sklearn.model_selection import train_test_split

dataset_pd = dataset_df.to_pandas()

train_pd, temp_pd = train_test_split(
    dataset_pd,
    test_size=0.20,
    random_state=42,
    stratify=dataset_pd["label"],
)

val_pd, test_pd = train_test_split(
    temp_pd,
    test_size=0.50,
    random_state=42,
    stratify=temp_pd["label"],
)

print("Train:", train_pd.shape)
print("Validation:", val_pd.shape)
print("Test:", test_pd.shape)

print()
print("Train labels:")
print(train_pd["label"].value_counts())

print()
print("Validation labels:")
print(val_pd["label"].value_counts())

print()
print("Test labels:")
print(test_pd["label"].value_counts())


## 10. TF-IDF + Logistic Regression modelini eğitme

Metinler kelime tabanlı unigram ve bigram TF-IDF özelliklerine dönüştürülür. Ardından dengeli sınıf ağırlıkları kullanan Logistic Regression modeli eğitilir.


In [ ]:
# TF-IDF özellik çıkarımı ve Logistic Regression sınıflandırıcısından oluşan modeli eğitir.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train = train_pd["text"]
y_train = train_pd["label"]

X_val = val_pd["text"]
y_val = val_pd["label"]

X_test = test_pd["text"]
y_test = test_pd["label"]

real_fake_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=150_000,
            sublinear_tf=True,
            lowercase=True,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42,
        ),
    ),
])

real_fake_model.fit(
    X_train,
    y_train,
)

print("Model eğitildi.")


## 11. Model değerlendirmesi

Validation ve test kümelerinde Accuracy, Precision, Recall, F1, Macro F1, ROC-AUC, classification report ve confusion matrix hesaplanır.


In [ ]:
# Sınıflandırma performansını farklı metriklerle ölçen yardımcı fonksiyonu tanımlar.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)


def evaluate_model(
    model,
    X,
    y,
    split_name,
):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    print("=" * 70)
    print(split_name)
    print("=" * 70)

    print(
        f"Accuracy:  "
        f"{accuracy_score(y, predictions):.4f}"
    )

    print(
        f"Precision: "
        f"{precision_score(y, predictions, zero_division=0):.4f}"
    )

    print(
        f"Recall:    "
        f"{recall_score(y, predictions, zero_division=0):.4f}"
    )

    print(
        f"F1:        "
        f"{f1_score(y, predictions, zero_division=0):.4f}"
    )

    print(
        f"Macro F1:  "
        f"{f1_score(y, predictions, average='macro'):.4f}"
    )

    print(
        f"ROC-AUC:   "
        f"{roc_auc_score(y, probabilities):.4f}"
    )

    print("\nClassification report:")

    print(
        classification_report(
            y,
            predictions,
            target_names=[
                "ANKA gerçek",
                "DMM yanlış",
            ],
            digits=4,
            zero_division=0,
        )
    )

    print("Confusion matrix:")

    print(
        confusion_matrix(
            y,
            predictions,
        )
    )

    return {
        "predictions": predictions,
        "probabilities": probabilities,
    }


In [ ]:
# Eğitilmiş modeli validation kümesinde değerlendirir.
val_results = evaluate_model(
    real_fake_model,
    X_val,
    y_val,
    "Validation",
)


In [ ]:
# Eğitilmiş modeli test kümesinde değerlendirir.
test_results = evaluate_model(
    real_fake_model,
    X_test,
    y_test,
    "Test",
)


## 12. Confusion matrix görselleştirmesi

Test kümesindeki doğru ve yanlış sınıflandırmalar görselleştirilir.


In [ ]:
# Test sonuçlarının confusion matrix grafiğini çizer.
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_results["predictions"],
    display_labels=[
        "ANKA gerçek",
        "DMM yanlış",
    ],
    values_format="d",
)

plt.title(
    "ANKA vs DMM Test Confusion Matrix"
)

plt.show()


## 13. Yeni bir metin üzerinde tahmin

Modelin `predict_proba` çıktısı kullanılarak metin:

- DMM-benzeri yanlış iddia
- ANKA-benzeri gerçek haber
- Belirsiz

olarak raporlanır. Eşik değerleri mevcut notebooktaki haliyle korunmuştur.


In [ ]:
# Yeni bir metin için olasılıkları ve belirsizlikli sınıf sonucunu üretir.
def predict_claim(text: str):
    text = str(text).strip()

    if not text:
        raise ValueError("Boş metin verilemez.")

    probabilities = real_fake_model.predict_proba([text])[0]
    fake_prob = float(probabilities[1])
    real_prob = float(probabilities[0])

    REAL_THRESHOLD = 0.20
    FAKE_THRESHOLD = 0.34

    if fake_prob >= FAKE_THRESHOLD:
        label = "DMM-benzeri yanlış iddia"
    elif fake_prob <= REAL_THRESHOLD:
        label = "ANKA-benzeri gerçek haber"
    else:
        label = "Belirsiz"

    print("Sonuç:", label)
    print(f"ANKA-benzeri olasılık: %{real_prob * 100:.2f}")
    print(f"DMM-benzeri yanlış iddia olasılığı: %{fake_prob * 100:.2f}")

    return {
        "label": label,
        "real_probability": real_prob,
        "false_claim_probability": fake_prob,
    }


## 14. Örnek tahmin

Aşağıdaki örnek metin eğitilmiş modele gönderilir.


In [ ]:
# Örnek bir iddiayı eğitilmiş modelle sınıflandırır.
text = (
    "Emeklilere her ay ücretsiz 100 litre akaryakıt desteği verileceği öne sürüldü. "
    "Sosyal medyada paylaşılan mesajlarda desteğin e-Devlet üzerinden otomatik olarak "
    "tanımlanacağı ve tüm akaryakıt istasyonlarında kullanılabileceği iddia edildi. "
    "Başvuruların gelecek hafta başlayacağı belirtildi. Ancak mesajlarda resmî bir bağlantı, "
    "karar metni veya yetkili kurum açıklaması yer almadı."
)

predict_claim(text)


## 15. Modeli Kaydetme

Model google drive'a kaydedilir.


In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = Path("/content/anka_dmm_tfidf_logistic_regression.joblib")

joblib.dump(
    real_fake_model,
    MODEL_PATH,
)

print("Model kaydedildi:", MODEL_PATH)
print(f"Dosya boyutu: {MODEL_PATH.stat().st_size / 1024**2:.2f} MB")

In [ ]:
DRIVE_MODEL_PATH = Path(
    "/content/drive/MyDrive/anka_dmm_tfidf_logistic_regression.joblib"
)

joblib.dump(
    real_fake_model,
    DRIVE_MODEL_PATH,
)

print("Model Drive'a kaydedildi:", DRIVE_MODEL_PATH)